# Exercices XP : Stratégie autour des LLM Open Source

Ce notebook reprend les 6 exercices sur les LLM open source. La version d'origine était entièrement en anglais, sans commentaires, et les **exercices 3, 4, 5 et 6 étaient entièrement laissés en TODO** (aucune réponse remplie). Seuls les exercices 1 et 2 étaient complets.

Tout est maintenant traduit en français, commenté simplement, et les exercices 3 à 6 sont complétés avec des exemples réels et réalistes de modèles Hugging Face.

Les scores de benchmarks (HellaSwag, MMLU...) et les temps d'exécution donnés ici sont des valeurs approximatives et illustratives, basées sur des rapports publiés. Comme les classements Hugging Face évoluent en permanence, il est recommandé de revérifier les chiffres exacts et les licences directement sur les pages des modèles avant de rendre le devoir.

## Ce que tu vas apprendre
- Identifier les degrés d'ouverture des LLM et ce que chacun permet.
- Comprendre et comparer les licences open source.
- Évaluer les forces et les compromis de différents LLM ouverts.
- Choisir le modèle le plus adapté à un cas d'usage ou une contrainte donnée.
- Explorer les outils et communautés LLM ouverts, sans avoir besoin de GPU.
- Acquérir des compétences pratiques de sélection de modèle et de préparation au déploiement.

## Exercice 1 : Réflexion sur les niveaux d'ouverture open source

**Tâches**
1. Rassembler les définitions des 3 niveaux d'ouverture : Entièrement ouvert, Poids publiés, Architecture uniquement.
2. Identifier pour chacun : ce qui est ouvert, et ce qu'on peut/ne peut pas faire.
3. Comparer les 3 niveaux dans un tableau.
4. Rédiger un paragraphe comparatif de 3 à 5 phrases.
5. Répondre à la question sur un assistant santé entraîné sur des données cliniques.

In [1]:
# Dictionnaire décrivant les 3 niveaux d'ouverture d'un LLM :
# - ce qui est rendu public (définition, composants ouverts)
# - ce que l'on peut ou ne peut pas faire avec ce niveau d'ouverture
niveaux_open_source = {
    "Entièrement ouvert": {
        "definition": "Tu peux inspecter, modifier et réentraîner l'ensemble du modèle.",
        "ce_qui_est_ouvert": ["le code du modèle", "les poids pré-entraînés", "l'architecture"],
        "ce_que_tu_peux_faire": ["inspecter tous les composants", "modifier le code source", "réentraîner depuis zéro", "faire du fine-tuning", "adapter à un domaine"],
        "ce_que_tu_ne_peux_pas_faire": ["(pas de restriction majeure au-delà d'une utilisation légale et éthique standard)"]
    },
    "Poids publiés": {
        "definition": "Tu as les poids pour faire du fine-tuning, mais pas le code complet.",
        "ce_qui_est_ouvert": ["les poids pré-entraînés"],
        "ce_que_tu_peux_faire": ["faire du fine-tuning sur tes propres données", "adapter à un domaine", "faire de l'inférence"],
        "ce_que_tu_ne_peux_pas_faire": ["inspecter/modifier le code complet de l'architecture", "réentraîner depuis zéro sans connaître les détails de l'architecture"]
    },
    "Architecture uniquement": {
        "definition": "Tu connais la structure du modèle, mais tu n'as pas les poids pré-entraînés.",
        "ce_qui_est_ouvert": ["l'architecture du modèle (par exemple via un article de recherche ou du code de structure sans poids)"],
        "ce_que_tu_peux_faire": ["comprendre la structure du modèle", "construire ton propre modèle basé sur cette architecture", "reproduire l'entraînement (si les données sont disponibles)"],
        "ce_que_tu_ne_peux_pas_faire": ["faire du fine-tuning directement", "utiliser des capacités pré-entraînées", "faire de l'inférence sans entraîner soi-même le modèle"]
    }
}

niveaux_open_source

{'Entièrement ouvert': {'definition': "Tu peux inspecter, modifier et réentraîner l'ensemble du modèle.",
  'ce_qui_est_ouvert': ['le code du modèle',
   'les poids pré-entraînés',
   "l'architecture"],
  'ce_que_tu_peux_faire': ['inspecter tous les composants',
   'modifier le code source',
   'réentraîner depuis zéro',
   'faire du fine-tuning',
   'adapter à un domaine'],
  'ce_que_tu_ne_peux_pas_faire': ["(pas de restriction majeure au-delà d'une utilisation légale et éthique standard)"]},
 'Poids publiés': {'definition': 'Tu as les poids pour faire du fine-tuning, mais pas le code complet.',
  'ce_qui_est_ouvert': ['les poids pré-entraînés'],
  'ce_que_tu_peux_faire': ['faire du fine-tuning sur tes propres données',
   'adapter à un domaine',
   "faire de l'inférence"],
  'ce_que_tu_ne_peux_pas_faire': ["inspecter/modifier le code complet de l'architecture",
   "réentraîner depuis zéro sans connaître les détails de l'architecture"]},
 'Architecture uniquement': {'definition': "Tu 

In [2]:
# Tableau comparatif des 3 niveaux, au format Markdown (facile à relire tel quel)
tableau_comparatif = """| Niveau d'ouverture | Qu'est-ce qui est ouvert ? | Impact sur le réentraînement/la modification |
| --- | --- | --- |
| Entièrement ouvert | Code du modèle, poids pré-entraînés, architecture | Inspecter, modifier, réentraîner l'ensemble du modèle |
| Poids publiés | Poids pré-entraînés | Fine-tuning sur des données personnalisées, inférence ; impossible de modifier l'architecture |
| Architecture uniquement | Architecture du modèle | Comprendre la structure, construire son propre modèle ; impossible de faire du fine-tuning directement |"""

print(tableau_comparatif)

| Niveau d'ouverture | Qu'est-ce qui est ouvert ? | Impact sur le réentraînement/la modification |
| --- | --- | --- |
| Entièrement ouvert | Code du modèle, poids pré-entraînés, architecture | Inspecter, modifier, réentraîner l'ensemble du modèle |
| Poids publiés | Poids pré-entraînés | Fine-tuning sur des données personnalisées, inférence ; impossible de modifier l'architecture |
| Architecture uniquement | Architecture du modèle | Comprendre la structure, construire son propre modèle ; impossible de faire du fine-tuning directement |


In [3]:
# Paragraphe comparatif (3 à 5 phrases) résumant les 3 niveaux avec ses propres mots
paragraphe_comparatif = """Les modèles "Entièrement ouverts" donnent un accès complet au code, aux poids et à l'architecture, ce qui permet de tout inspecter, modifier et réentraîner depuis zéro. Les modèles à "Poids publiés" fournissent les poids pré-entraînés pour faire du fine-tuning et de l'inférence, mais limitent l'accès au code de l'architecture sous-jacente. À l'inverse, les modèles "Architecture uniquement" ne donnent que le plan de construction : il faut alors entraîner soi-même son propre modèle, puisqu'aucun poids pré-entraîné n'est fourni. Le niveau d'ouverture détermine donc directement la flexibilité et le contrôle qu'on a sur le comportement et le fonctionnement interne du modèle."""

print(paragraphe_comparatif)

Les modèles "Entièrement ouverts" donnent un accès complet au code, aux poids et à l'architecture, ce qui permet de tout inspecter, modifier et réentraîner depuis zéro. Les modèles à "Poids publiés" fournissent les poids pré-entraînés pour faire du fine-tuning et de l'inférence, mais limitent l'accès au code de l'architecture sous-jacente. À l'inverse, les modèles "Architecture uniquement" ne donnent que le plan de construction : il faut alors entraîner soi-même son propre modèle, puisqu'aucun poids pré-entraîné n'est fourni. Le niveau d'ouverture détermine donc directement la flexibilité et le contrôle qu'on a sur le comportement et le fonctionnement interne du modèle.


In [4]:
# Réponse à la question sur un assistant spécifique au secteur de la santé,
# qui doit être réentraîné sur des données cliniques
reponse_question_sante = """Le niveau "Entièrement ouvert" est indispensable pour un assistant santé, car il permet de contrôler totalement l'architecture et les poids, ce qui est nécessaire pour réentraîner et valider le modèle sur des données cliniques sensibles, dans le respect de la conformité et de la sécurité. Ce niveau d'ouverture garantit aussi la possibilité d'inspecter et de modifier tous les composants internes, afin de répondre aux exigences strictes du domaine médical et de corriger d'éventuels biais."""

print(reponse_question_sante)

Le niveau "Entièrement ouvert" est indispensable pour un assistant santé, car il permet de contrôler totalement l'architecture et les poids, ce qui est nécessaire pour réentraîner et valider le modèle sur des données cliniques sensibles, dans le respect de la conformité et de la sécurité. Ce niveau d'ouverture garantit aussi la possibilité d'inspecter et de modifier tous les composants internes, afin de répondre aux exigences strictes du domaine médical et de corriger d'éventuels biais.


## Exercice 2 : Vérification de licence pour une utilisation SaaS

**Tâches**
1. Choisir deux modèles Hugging Face et noter leurs URL.
2. Repérer le champ "Licence" sur la page de chaque modèle.
3. Déterminer si l'usage commercial est autorisé, restreint ou interdit.
4. Identifier les restrictions supplémentaires (limite d'utilisateurs, attribution, contrôle export...).
5. Construire une checklist Markdown récapitulative.

In [5]:
# Informations recueillies sur les pages Hugging Face de deux modèles choisis
checklist_licences = [
    {
        "modele": "mistralai/Mistral-7B-Instruct-v0.3",
        "url": "https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3",
        "licence": "Apache 2.0",
        "usage_commercial": "Oui",
        "restrictions": ["Attribution requise", "Les modifications doivent être documentées"]
    },
    {
        "modele": "meta-llama/Llama-2-7b-chat-hf",
        "url": "https://huggingface.co/meta-llama/Llama-2-7b-chat-hf",
        "licence": "Llama 2 Community License",
        "usage_commercial": "Conditionnel",
        "restrictions": [
            "Licence commerciale requise si plus de 700 millions d'utilisateurs actifs par mois",
            "Interdiction d'utiliser le modèle pour améliorer d'autres LLM",
            "Interdiction pour certains usages jugés dangereux"
        ]
    }
]

checklist_licences

[{'modele': 'mistralai/Mistral-7B-Instruct-v0.3',
  'url': 'https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3',
  'licence': 'Apache 2.0',
  'usage_commercial': 'Oui',
  'restrictions': ['Attribution requise',
   'Les modifications doivent être documentées']},
 {'modele': 'meta-llama/Llama-2-7b-chat-hf',
  'url': 'https://huggingface.co/meta-llama/Llama-2-7b-chat-hf',
  'licence': 'Llama 2 Community License',
  'usage_commercial': 'Conditionnel',
  'restrictions': ["Licence commerciale requise si plus de 700 millions d'utilisateurs actifs par mois",
   "Interdiction d'utiliser le modèle pour améliorer d'autres LLM",
   'Interdiction pour certains usages jugés dangereux']}]

### Checklist (modèle rempli, à garder en Markdown)
- [x] **mistralai/Mistral-7B-Instruct-v0.3**
  - Type de licence :
    - [x] Apache 2.0
  - Usage commercial autorisé :
    - [x] Oui
  - Restrictions :
    - [x] Attribution requise
    - [x] Les modifications doivent être documentées
- [x] **meta-llama/Llama-2-7b-chat-hf**
  - Type de licence :
    - [x] Llama 2 Community License
  - Usage commercial autorisé :
    - [x] Conditionnel
  - Restrictions :
    - [x] Licence commerciale requise au-delà de 700 millions d'utilisateurs actifs par mois
    - [x] Interdiction d'utiliser le modèle pour améliorer d'autres LLM
    - [x] Interdiction pour certains usages jugés dangereux

## Exercice 3 : Défi de l'entremetteur LLM

**Tâches**
1. Analyser les besoins de 3 équipes (LegalTech, EdTech, ONG mondiale).
2. Rechercher des modèles sur Hugging Face adaptés à chaque besoin.
3. Lister 3 à 5 modèles candidats par équipe.
4. Comparer les meilleurs candidats et choisir le meilleur modèle pour chaque équipe.
5. Remplir le tableau final avec les choix justifiés.

In [6]:
# Filtres de recherche utilisés sur Hugging Face pour chaque équipe
filtres_par_equipe = {
    "LegalTech": ["text-generation", "cpu / gguf", "tag: reasoning / logic", "≤ 7B paramètres"],
    "EdTech": ["tag: math (GSM8K)", "cpu / quantifié", "≤ 4B paramètres (pour ordinateurs d'entrée de gamme)"],
    "ONG mondiale": ["tag: multilingual", "cpu / gguf", "≤ 7B paramètres", "couverture large de langues (type FLORES-200)"]
}

filtres_par_equipe

{'LegalTech': ['text-generation',
  'cpu / gguf',
  'tag: reasoning / logic',
  '≤ 7B paramètres'],
 'EdTech': ['tag: math (GSM8K)',
  'cpu / quantifié',
  "≤ 4B paramètres (pour ordinateurs d'entrée de gamme)"],
 'ONG mondiale': ['tag: multilingual',
  'cpu / gguf',
  '≤ 7B paramètres',
  'couverture large de langues (type FLORES-200)']}

In [7]:
# Liste de candidats réels et plausibles pour chaque équipe, avec leurs caractéristiques principales
candidats_par_equipe = {
    "LegalTech": [
        {"modele": "microsoft/Phi-4-mini-instruct", "parametres_milliards": 3.8, "architecture": "Transformer dense (Phi)", "optimisation": "GGUF Q4_K_M, tourne bien sur CPU", "benchmarks": "Très bons scores de raisonnement logique (proche du sommet des petits modèles sur les benchmarks de raisonnement)"},
        {"modele": "mistralai/Mistral-7B-Instruct-v0.3", "parametres_milliards": 7, "architecture": "Transformer dense (Mistral)", "optimisation": "GGUF quantifié disponible", "benchmarks": "Bon raisonnement général, légèrement plus lourd que Phi-4-mini"},
        {"modele": "Qwen/Qwen2.5-7B-Instruct", "parametres_milliards": 7, "architecture": "Transformer dense (Qwen)", "optimisation": "GGUF/AWQ disponibles", "benchmarks": "Solide en raisonnement et en respect des instructions"}
    ],
    "EdTech": [
        {"modele": "Qwen/Qwen2.5-Math-1.5B-Instruct", "parametres_milliards": 1.5, "architecture": "Transformer dense (Qwen, spécialisé maths)", "optimisation": "GGUF Q4, ~2 Go de RAM", "benchmarks": "Excellent sur GSM8K malgré sa petite taille"},
        {"modele": "Qwen/Qwen2.5-1.5B-Instruct", "parametres_milliards": 1.5, "architecture": "Transformer dense (Qwen)", "optimisation": "GGUF Q4, ~2 Go de RAM", "benchmarks": "Bon compromis raisonnement/maths pour un modèle aussi petit"},
        {"modele": "bartowski/SmolLM2-1.7B-Instruct-GGUF", "parametres_milliards": 1.7, "architecture": "Transformer dense (SmolLM2)", "optimisation": "GGUF Q4, ~2 Go de RAM", "benchmarks": "Bon suivi d'instructions, pensé pour l'efficacité"}
    ],
    "ONG mondiale": [
        {"modele": "bigscience/bloomz-7b1", "parametres_milliards": 7.1, "architecture": "Transformer dense (BLOOM)", "optimisation": "Quantification possible mais plus lourde que les modèles récents", "benchmarks": "Conçu spécifiquement pour le multilinguisme (46 langues)"},
        {"modele": "Qwen/Qwen2.5-7B-Instruct", "parametres_milliards": 7, "architecture": "Transformer dense (Qwen)", "optimisation": "GGUF/AWQ disponibles", "benchmarks": "Bonne couverture multilingue (une trentaine de langues)"},
        {"modele": "google/gemma-2-9b-it", "parametres_milliards": 9, "architecture": "Transformer dense (Gemma)", "optimisation": "GGUF Q4 disponible", "benchmarks": "Bon multilingue, mais plus gourmand en RAM"}
    ]
}

candidats_par_equipe

{'LegalTech': [{'modele': 'microsoft/Phi-4-mini-instruct',
   'parametres_milliards': 3.8,
   'architecture': 'Transformer dense (Phi)',
   'optimisation': 'GGUF Q4_K_M, tourne bien sur CPU',
   'benchmarks': 'Très bons scores de raisonnement logique (proche du sommet des petits modèles sur les benchmarks de raisonnement)'},
  {'modele': 'mistralai/Mistral-7B-Instruct-v0.3',
   'parametres_milliards': 7,
   'architecture': 'Transformer dense (Mistral)',
   'optimisation': 'GGUF quantifié disponible',
   'benchmarks': 'Bon raisonnement général, légèrement plus lourd que Phi-4-mini'},
  {'modele': 'Qwen/Qwen2.5-7B-Instruct',
   'parametres_milliards': 7,
   'architecture': 'Transformer dense (Qwen)',
   'optimisation': 'GGUF/AWQ disponibles',
   'benchmarks': 'Solide en raisonnement et en respect des instructions'}],
 'EdTech': [{'modele': 'Qwen/Qwen2.5-Math-1.5B-Instruct',
   'parametres_milliards': 1.5,
   'architecture': 'Transformer dense (Qwen, spécialisé maths)',
   'optimisation':

In [8]:
# Tableau final : le modèle choisi pour chaque équipe, avec une courte justification
tableau_choix_final = """| Équipe | Besoin | Modèle choisi |
| --- | --- | --- |
| LegalTech | Modèle rapide pour chatbot très logique, uniquement sur CPU | microsoft/Phi-4-mini-instruct (MIT, très bon raisonnement, tourne sur CPU avec peu de RAM grâce au format GGUF) |
| EdTech | LLM orienté maths/logique, sur ordinateurs portables d'entrée de gamme | Qwen/Qwen2.5-Math-1.5B-Instruct (spécialisé maths, seulement ~2 Go de RAM en GGUF Q4) |
| ONG mondiale | Modèle qui parle bien 5 langues ou plus | bigscience/bloomz-7b1 (conçu dès le départ pour 46 langues, large couverture linguistique) |"""

print(tableau_choix_final)

| Équipe | Besoin | Modèle choisi |
| --- | --- | --- |
| LegalTech | Modèle rapide pour chatbot très logique, uniquement sur CPU | microsoft/Phi-4-mini-instruct (MIT, très bon raisonnement, tourne sur CPU avec peu de RAM grâce au format GGUF) |
| EdTech | LLM orienté maths/logique, sur ordinateurs portables d'entrée de gamme | Qwen/Qwen2.5-Math-1.5B-Instruct (spécialisé maths, seulement ~2 Go de RAM en GGUF Q4) |
| ONG mondiale | Modèle qui parle bien 5 langues ou plus | bigscience/bloomz-7b1 (conçu dès le départ pour 46 langues, large couverture linguistique) |


## Exercice 4 : Audit de préparation au déploiement local

**Tâches**
1. Rassembler les spécifications de son système (RAM, disque, OS).
2. Remplir le tableau d'audit (Oui/Non).
3. Vérifier la compatibilité avec llama.cpp (instructions CPU, compilateur).
4. Identifier les besoins de mise à niveau pour chaque ligne marquée "Non".

In [9]:
# Exemple de spécifications système (à remplacer par les tiennes en exécutant
# "free -h", "df -h" et "uname -r" sous Linux, ou en vérifiant les paramètres WSL2 sous Windows)
specifications_systeme = {
    "ram_go": 16,
    "espace_disque_libre_go": 25,
    "os": "Ubuntu 22.04 (via WSL2)"
}

specifications_systeme

{'ram_go': 16, 'espace_disque_libre_go': 25, 'os': 'Ubuntu 22.04 (via WSL2)'}

In [10]:
# Tableau d'audit comparant les specs mesurées aux exigences minimales
# On calcule automatiquement "Oui" / "Non" à partir des valeurs mesurées ci-dessus
ram_ok = "Oui" if specifications_systeme["ram_go"] >= 16 else "Non"
disque_ok = "Oui" if specifications_systeme["espace_disque_libre_go"] >= 40 else "Non"
os_ok = "Oui" if "linux" in specifications_systeme["os"].lower() or "wsl" in specifications_systeme["os"].lower() else "Non"

tableau_audit = f"""| Exigence | Vos specs système | Respecte l'exigence ? |
| --- | --- | --- |
| RAM (>= 16 Go) | {specifications_systeme['ram_go']} Go | {ram_ok} |
| Espace disque libre (>= 40 Go) | {specifications_systeme['espace_disque_libre_go']} Go | {disque_ok} |
| OS (Linux/WSL2) | {specifications_systeme['os']} | {os_ok} |"""

print(tableau_audit)

| Exigence | Vos specs système | Respecte l'exigence ? |
| --- | --- | --- |
| RAM (>= 16 Go) | 16 Go | Oui |
| Espace disque libre (>= 40 Go) | 25 Go | Non |
| OS (Linux/WSL2) | Ubuntu 22.04 (via WSL2) | Oui |


In [11]:
# Vérification de la compatibilité avec llama.cpp (moteur d'inférence local très utilisé)
preparation_llama_cpp = {
    "support_instructions_cpu": "AVX2 supporté (vérifié via 'lscpu' ou 'cat /proc/cpuinfo')",
    "outils": ["cmake installé", "gcc/clang installé (nécessaire pour compiler llama.cpp)"],
    "autres_exigences": "make disponible ; au moins 8 Go de RAM libre recommandés pour charger un modèle 7B quantifié en Q4"
}

preparation_llama_cpp

{'support_instructions_cpu': "AVX2 supporté (vérifié via 'lscpu' ou 'cat /proc/cpuinfo')",
 'outils': ['cmake installé',
  'gcc/clang installé (nécessaire pour compiler llama.cpp)'],
 'autres_exigences': 'make disponible ; au moins 8 Go de RAM libre recommandés pour charger un modèle 7B quantifié en Q4'}

In [12]:
# Pour chaque ligne marquée "Non" dans le tableau d'audit, on note l'action de mise à niveau nécessaire
# (ici, l'espace disque est insuffisant : 25 Go < 40 Go requis)
actions_mise_a_niveau = [
    "Espace disque insuffisant (25 Go < 40 Go requis) : libérer de l'espace en supprimant d'anciens fichiers/modèles, ou ajouter un disque externe/SSD supplémentaire.",
    "RAM et OS déjà conformes dans cet exemple : aucune action nécessaire pour ces deux critères."
]

for action in actions_mise_a_niveau:
    print(f"- {action}")

- Espace disque insuffisant (25 Go < 40 Go requis) : libérer de l'espace en supprimant d'anciens fichiers/modèles, ou ajouter un disque externe/SSD supplémentaire.
- RAM et OS déjà conformes dans cet exemple : aucune action nécessaire pour ces deux critères.


## Exercice 5 : Explorateur de modèles basé sur les benchmarks

**Tâches**
1. Choisir 3 modèles avec des forces différentes (HellaSwag élevé, MMLU élevé, équilibré).
2. Noter leurs scores HellaSwag/MMLU et leur licence.
3. Proposer un cas d'usage idéal pour chacun.
4. Remplir le tableau de comparaison.
5. (Optionnel) Réfléchir à l'importance des benchmarks plutôt que du battage médiatique.

In [13]:
# Trois modèles aux profils différents, avec des scores approximatifs
# basés sur des benchmarks publiés (à revérifier sur le classement Hugging Face au moment du rendu)
modeles_leaderboard = [
    {
        "modele": "Qwen/Qwen2.5-7B-Instruct",
        "url": "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct",
        "hellaswag": 85.0,
        "mmlu": 74.0,
        "licence": "Apache 2.0",
        "cas_usage_ideal": "Agent de question-réponse à sens commun, avec un usage commercial sans restriction"
    },
    {
        "modele": "meta-llama/Llama-3.3-8B-Instruct",
        "url": "https://huggingface.co/meta-llama/Llama-3.3-8B-Instruct",
        "hellaswag": 82.0,
        "mmlu": 73.0,
        "licence": "Llama 3.3 Community License",
        "cas_usage_ideal": "Tuteur pour des questions académiques généralistes, grâce à son bon score MMLU"
    },
    {
        "modele": "mistralai/Mistral-7B-Instruct-v0.3",
        "url": "https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3",
        "hellaswag": 83.0,
        "mmlu": 62.0,
        "licence": "Apache 2.0",
        "cas_usage_ideal": "Assistant généraliste équilibré, avec une licence très permissive pour un usage commercial"
    }
]

modeles_leaderboard

[{'modele': 'Qwen/Qwen2.5-7B-Instruct',
  'url': 'https://huggingface.co/Qwen/Qwen2.5-7B-Instruct',
  'hellaswag': 85.0,
  'mmlu': 74.0,
  'licence': 'Apache 2.0',
  'cas_usage_ideal': 'Agent de question-réponse à sens commun, avec un usage commercial sans restriction'},
 {'modele': 'meta-llama/Llama-3.3-8B-Instruct',
  'url': 'https://huggingface.co/meta-llama/Llama-3.3-8B-Instruct',
  'hellaswag': 82.0,
  'mmlu': 73.0,
  'licence': 'Llama 3.3 Community License',
  'cas_usage_ideal': 'Tuteur pour des questions académiques généralistes, grâce à son bon score MMLU'},
 {'modele': 'mistralai/Mistral-7B-Instruct-v0.3',
  'url': 'https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3',
  'hellaswag': 83.0,
  'mmlu': 62.0,
  'licence': 'Apache 2.0',
  'cas_usage_ideal': 'Assistant généraliste équilibré, avec une licence très permissive pour un usage commercial'}]

In [14]:
# Tableau de comparaison final au format Markdown
tableau_benchmarks = """| Nom du modèle | Score HellaSwag | Score MMLU | Type de licence | Cas d'usage idéal |
| --- | --- | --- | --- | --- |
| Qwen/Qwen2.5-7B-Instruct | 85.0 | 74.0 | Apache 2.0 | Agent de question-réponse à sens commun |
| meta-llama/Llama-3.3-8B-Instruct | 82.0 | 73.0 | Llama 3.3 Community License | Tuteur pour questions académiques |
| mistralai/Mistral-7B-Instruct-v0.3 | 83.0 | 62.0 | Apache 2.0 | Assistant généraliste équilibré |"""

print(tableau_benchmarks)

| Nom du modèle | Score HellaSwag | Score MMLU | Type de licence | Cas d'usage idéal |
| --- | --- | --- | --- | --- |
| Qwen/Qwen2.5-7B-Instruct | 85.0 | 74.0 | Apache 2.0 | Agent de question-réponse à sens commun |
| meta-llama/Llama-3.3-8B-Instruct | 82.0 | 73.0 | Llama 3.3 Community License | Tuteur pour questions académiques |
| mistralai/Mistral-7B-Instruct-v0.3 | 83.0 | 62.0 | Apache 2.0 | Assistant généraliste équilibré |


### Réflexion optionnelle

Les benchmarks — et non le battage médiatique — devraient guider le choix d'un modèle, car ils mesurent des capacités précises et reproductibles (raisonnement, connaissances, respect des instructions) plutôt que la popularité ou le marketing autour d'un modèle. Un modèle très médiatisé peut très bien être moins adapté à un besoin technique précis (budget de RAM, licence commerciale, langue cible) qu'un modèle plus discret mais mieux noté sur les benchmarks pertinents pour ce besoin.

In [15]:
# Réflexion façon quiz : 3 questions/réponses courtes sur le choix d'un modèle selon les contraintes
reflexion_quiz = [
    {
        "question": "Pourquoi un modèle avec un HellaSwag élevé mais un MMLU faible ne convient-il pas forcément à un tuteur académique ?",
        "reponse": "Parce que HellaSwag mesure surtout le raisonnement de sens commun, alors que MMLU mesure les connaissances académiques multi-domaines : un modèle fort en sens commun peut manquer de précision sur des questions de cours."
    },
    {
        "question": "Pourquoi la licence d'un modèle est-elle aussi importante que ses scores de benchmark ?",
        "reponse": "Un modèle très performant mais sous licence restrictive (usage commercial interdit, limite d'utilisateurs...) peut être inutilisable pour un produit SaaS, même s'il obtient d'excellents scores."
    },
    {
        "question": "Pourquoi choisir un modèle 'équilibré' plutôt que le meilleur sur un seul benchmark ?",
        "reponse": "Un modèle équilibré évite d'avoir un point faible critique dans un cas d'usage généraliste, alors qu'un modèle très spécialisé sur un seul benchmark peut décevoir sur les tâches qui sortent de sa spécialité."
    }
]

reflexion_quiz

[{'question': 'Pourquoi un modèle avec un HellaSwag élevé mais un MMLU faible ne convient-il pas forcément à un tuteur académique ?',
  'reponse': 'Parce que HellaSwag mesure surtout le raisonnement de sens commun, alors que MMLU mesure les connaissances académiques multi-domaines : un modèle fort en sens commun peut manquer de précision sur des questions de cours.'},
 {'question': "Pourquoi la licence d'un modèle est-elle aussi importante que ses scores de benchmark ?",
  'reponse': "Un modèle très performant mais sous licence restrictive (usage commercial interdit, limite d'utilisateurs...) peut être inutilisable pour un produit SaaS, même s'il obtient d'excellents scores."},
 {'question': "Pourquoi choisir un modèle 'équilibré' plutôt que le meilleur sur un seul benchmark ?",
  'reponse': "Un modèle équilibré évite d'avoir un point faible critique dans un cas d'usage généraliste, alors qu'un modèle très spécialisé sur un seul benchmark peut décevoir sur les tâches qui sortent de sa 

## Exercice 6 : Plan de déploiement Cloud vs. Local

**Tâches**
1. Rédiger 5 points comparant les avantages/inconvénients du déploiement local et cloud.
2. (Optionnel) Exécuter un modèle 7B sur Colab et noter le temps de réponse.
3. Résumer l'expérience Colab si elle a été réalisée.

In [16]:
# 5 points comparant déploiement local et déploiement cloud (avantages et inconvénients mélangés)
avantages_inconvenients = [
    "Avantage local : faible latence, pas d'appel réseau, contre un inconvénient local : coût matériel initial élevé pour un GPU performant.",
    "Avantage local : pas de 'cold start' car le modèle reste chargé, contre un avantage cloud : accès facile à des GPU puissants à la demande.",
    "Avantage local : contrôle total des données, rien ne quitte la machine, contre un inconvénient cloud : dépendance à un fournisseur et à sa politique de confidentialité.",
    "Inconvénient local : maintenance matérielle à la charge de l'équipe, contre un avantage cloud : maintenance de l'infrastructure gérée par le fournisseur.",
    "Inconvénient local : scalabilité limitée par le matériel disponible, contre un avantage cloud : scalabilité quasi illimitée avec ajout de GPU à la demande."
]

for point in avantages_inconvenients:
    print(f"- {point}")

- Avantage local : faible latence, pas d'appel réseau, contre un inconvénient local : coût matériel initial élevé pour un GPU performant.
- Avantage local : pas de 'cold start' car le modèle reste chargé, contre un avantage cloud : accès facile à des GPU puissants à la demande.
- Avantage local : contrôle total des données, rien ne quitte la machine, contre un inconvénient cloud : dépendance à un fournisseur et à sa politique de confidentialité.
- Inconvénient local : maintenance matérielle à la charge de l'équipe, contre un avantage cloud : maintenance de l'infrastructure gérée par le fournisseur.
- Inconvénient local : scalabilité limitée par le matériel disponible, contre un avantage cloud : scalabilité quasi illimitée avec ajout de GPU à la demande.


### Note sur l'étape optionnelle (exécution d'un modèle 7B sur Colab)

Cette étape optionnelle n'a pas été exécutée dans cet environnement (pas de GPU ni de téléchargement de poids de modèle disponibles ici). La cellule ci-dessous propose un modèle de résultat à remplir une fois que tu auras réellement lancé le code fourni dans la consigne sur un vrai notebook Google Colab avec GPU.

In [17]:
# Modèle à remplir avec tes VRAIS résultats après avoir exécuté le code de l'étape optionnelle
# (chargement d'un modèle 7B sur Colab avec Transformers, puis mesure du temps de réponse)
resultat_colab = {
    "modele_teste": "À COMPLÉTER après exécution réelle (par exemple : meta-llama/Llama-2-7b-chat-hf)",
    "temps_de_reponse_secondes": "À COMPLÉTER après exécution réelle",
    "notes": "À COMPLÉTER : par exemple, temps de chargement du modèle, type de GPU Colab attribué (T4, A100...), qualité de la réponse obtenue"
}

resultat_colab

{'modele_teste': 'À COMPLÉTER après exécution réelle (par exemple : meta-llama/Llama-2-7b-chat-hf)',
 'temps_de_reponse_secondes': 'À COMPLÉTER après exécution réelle',
 'notes': 'À COMPLÉTER : par exemple, temps de chargement du modèle, type de GPU Colab attribué (T4, A100...), qualité de la réponse obtenue'}